# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR^2 Croissant dataset using the `mlcroissant` library—all while referencing fields, columns, and record sets by their `@id` as required by the Croissant specification.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata via Croissant
dataset = mlc.Dataset(croissant_url)

# Print dataset high-level info
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Version: {getattr(meta, 'version', 'N/A')}")
print(f"Keywords: {getattr(meta, 'keywords', [])}")

## 2. Data Overview
Review available record sets (`cr:RecordSet` entities), fields (`cr:Field`), and column IDs. All are referenced by their Croissant `@id`.

In [ ]:
print("Available record sets (by @id):")
all_record_sets = dataset.record_sets
for rs in all_record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields (@id and name):")
        for f in rs.fields:
            print(f"    - {f.id} ({f.name})")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns (@id and name):")
        for col in rs.columns:
            print(f"    - {col.id} ({col.name})")
    print()

## 3. Data Extraction
Let's extract data from a record set of interest (referenced only by its `@id`). Here, we choose the main protein abundance record set typically containing values such as 'accession', 'description', 'abundance', 'peptides', etc.

Change the `record_set_ids` or choose additional ones as desired. Below code will load all available record sets into pandas DataFrames, keyed by their `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for RecordSet {record_set_id}\n")

# To demonstrate, select the main record set
main_record_set_id = None
if record_set_ids:
    # Heuristic: choose the record set that contains 'protein' or 'abundance' as most likely the main table
    for rs in dataset.record_sets:
        if 'protein' in (rs.name or '').lower() or 'abundance' in (rs.name or '').lower():
            main_record_set_id = rs.id
            break
    # Fallback: just choose the first one
    if not main_record_set_id:
        main_record_set_id = record_set_ids[0]

# List columns and preview records
if main_record_set_id:
    print(f"Columns in main RecordSet {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print('No record sets found.')

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group by fields, referencing them by their unique `@id` as specified in the Croissant Schema.

> Adjust `numeric_field_id` and `group_field_id` as desired based on the previous overview.

In [ ]:
# Set your field @ids here (from the field overview above)
# Example (replace as appropriate):
numeric_field_id = None
group_field_id = None

# Heuristic: try common intensity/abundance/mass/pI/peptide column names
main_cols = dataframes[main_record_set_id].columns if main_record_set_id else []
# Typical candidates for numeric analysis: 'abundance', 'peptides', 'MW', 'Score', etc.
for candidate in main_cols:
    if any(word in candidate.lower() for word in ['abundance', 'score', 'peptide', 'mw', 'pi', 'coverage']):
        numeric_field_id = candidate
        break
if not numeric_field_id:
    # Fallback: use the first numeric-like column
    for col in main_cols:
        if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][col]):
            numeric_field_id = col
            break
if not numeric_field_id:
    print("No clear numeric field found. Please pick a numeric column from the above list.")

# Try to pick a grouping categorical field (e.g., 'sample', 'cell_type', etc.)
for candidate in main_cols:
    if any(word in candidate.lower() for word in ['sample', 'cell', 'group']):
        group_field_id = candidate
        break

df = dataframes[main_record_set_id]
if numeric_field_id and numeric_field_id in df.columns:
    # Remove records where value is missing or non-numeric
    df = df.copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.75)  # set threshold as 75th percentile
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (field @id: {numeric_field_id}): {filtered_df.shape[0]} rows")

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nPreview of normalized '{numeric_field_id}':")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped statistics by '{group_field_id}':")
        display(grouped_df.head())
else:
    print('No suitable numeric field for EDA found.')

## 5. Visualization
Visualize data distributions or field relationships. We'll plot histogram and boxplot for the chosen numeric field, and if a grouping field is available, a group-level bar chart.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")

    plt.subplot(1,2,2)
    sns.boxplot(y=df[numeric_field_id].dropna())
    plt.title(f"Boxplot of '{numeric_field_id}'")
    plt.tight_layout()
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        group_avg = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(12)
        sns.barplot(y=group_avg.index, x=group_avg.values, orient='h')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel(group_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated a FAIR Croissant dataset workflow using `mlcroissant`, including referencing all entities by `@id`.

- You can now select other record sets, fields, or perform custom analyses as needed.
- All steps here were made referencing only Croissant `@id`s for robust, schema-anchored data science.